# LeetCode 2297 — Jump Game VIII bằng Python

Link đề: https://leetcode.com/problems/jump-game-viii/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Stack, Graph, Monotonic Stack, Shortest Path

> **Bài Premium:** đề bài trong notebook được viết theo hiểu biết về bài (nguồn doocs). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc và chữ ký hàm.

Skeleton LeetCode:

```python
class Solution:
    def minCost(self, nums: List[int], costs: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Đọc kỹ điều kiện nhảy và rút ra rằng từ mỗi `i` **chỉ có tối đa hai** bước nhảy hợp lệ: tới **phần tử ≥ `nums[i]` đầu tiên** bên phải và tới **phần tử < `nums[i]` đầu tiên** bên phải.
2. Tính hai "phần tử kế tiếp" đó cho mọi `i` trong O(n) bằng **ngăn xếp đơn điệu**.
3. Đồ thị nhảy là một **DAG** (chỉ nhảy sang phải) ⇒ đường đi chi phí nhỏ nhất bằng **DP theo thứ tự chỉ số**, không cần Dijkstra.
4. Gộp xây đồ thị và DP thành **một lượt** trái → phải: khi một chỉ số bị pop khỏi ngăn xếp, ta biết ngay cạnh của nó.
5. Kiểm chứng bằng vét cạn theo **đúng định nghĩa** trên mảng nhỏ ngẫu nhiên và tìm phản ví dụ cho các cách hiểu sai điều kiện.

## 2. Đề bài & ví dụ

Cho mảng `nums` độ dài `n`, ban đầu đứng ở chỉ số 0. Có thể nhảy từ `i` tới `j` (`i < j`) nếu:

- `nums[i] ≤ nums[j]` **và** `nums[k] < nums[i]` với mọi `i < k < j`; **hoặc**
- `nums[i] > nums[j]` **và** `nums[k] ≥ nums[i]` với mọi `i < k < j`.

Cho mảng `costs`, `costs[i]` là chi phí khi **nhảy tới** chỉ số `i`. Trả về chi phí nhỏ nhất để tới chỉ số `n − 1`.

**Ví dụ 1:** `nums = [3,2,4,4,1]`, `costs = [3,7,6,4,2]` → `8` (0 → 2 → 4: `6 + 2`).

**Ví dụ 2:** `nums = [0,1,2]`, `costs = [1,1,1]` → `2` (0 → 1 → 2; không nhảy thẳng 0 → 2 được vì `nums[1] = 1` không `< nums[0] = 0`).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`; `0 ≤ nums[i], costs[i] ≤ 10⁵`.

In [1]:
from typing import List
from collections import defaultdict
from math import inf
import heapq, random, time

examples = [([3, 2, 4, 4, 1], [3, 7, 6, 4, 2], 8), ([0, 1, 2], [1, 1, 1], 2)]
for a, c, exp in examples:
    print(f"nums={a} costs={c} -> kỳ vọng {exp}")

nums=[3, 2, 4, 4, 1] costs=[3, 7, 6, 4, 2] -> kỳ vọng 8
nums=[0, 1, 2] costs=[1, 1, 1] -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1 Kiểm tra điều kiện nhảy theo đúng định nghĩa

In [2]:
def can_jump(nums, i, j):
    mid = nums[i + 1:j]
    if nums[i] <= nums[j]:
        return all(x < nums[i] for x in mid)
    return all(x >= nums[i] for x in mid)

a = examples[0][0]
for i in range(len(a)):
    print(f"từ {i} (nums={a[i]}) nhảy được tới:", [j for j in range(i + 1, len(a)) if can_jump(a, i, j)])

từ 0 (nums=3) nhảy được tới: [1, 2]
từ 1 (nums=2) nhảy được tới: [2, 4]
từ 2 (nums=4) nhảy được tới: [3, 4]
từ 3 (nums=4) nhảy được tới: [4]
từ 4 (nums=1) nhảy được tới: []


### 3.2 Hai "phần tử kế tiếp" bằng ngăn xếp đơn điệu

- `nge[i]` = chỉ số `j > i` nhỏ nhất với `nums[j] ≥ nums[i]` (next greater-or-equal).
- `nse[i]` = chỉ số `j > i` nhỏ nhất với `nums[j] < nums[i]` (next strictly smaller).

Duyệt từ phải sang trái, giữ ngăn xếp các ứng viên; pop những phần tử **không thể** là đáp án cho `i` và cho mọi chỉ số bên trái `i`.

In [3]:
def next_ge(nums):
    n = len(nums); res = [-1] * n; st = []
    for i in range(n - 1, -1, -1):
        while st and nums[st[-1]] < nums[i]: st.pop()       # nhỏ hơn nums[i]: không phải ">= nums[i]"
        res[i] = st[-1] if st else -1
        st.append(i)
    return res

def next_lt(nums):
    n = len(nums); res = [-1] * n; st = []
    for i in range(n - 1, -1, -1):
        while st and nums[st[-1]] >= nums[i]: st.pop()      # >= nums[i]: không phải "< nums[i]"
        res[i] = st[-1] if st else -1
        st.append(i)
    return res

a = examples[0][0]
print("nums:", a); print("nge :", next_ge(a)); print("nlt :", next_lt(a))

nums: [3, 2, 4, 4, 1]
nge : [2, 2, 3, -1, -1]
nlt : [1, 4, 4, 4, -1]


### 3.3 Đường đi ngắn nhất trên DAG

Nếu mọi cạnh đi từ chỉ số nhỏ sang chỉ số lớn, thứ tự `0, 1, …, n−1` là một **thứ tự tô-pô**. Duyệt theo thứ tự đó và "nới lỏng" (relax) cạnh: `f[j] = min(f[j], f[i] + w(i, j))`. Khi tới `i`, `f[i]` đã tối ưu vì mọi cạnh vào `i` xuất phát từ chỉ số nhỏ hơn. Chi phí O(V + E), không cần hàng đợi ưu tiên.

### 3.4 Luôn nhảy được sang `i + 1`

Đoạn giữa rỗng ⇒ một trong hai điều kiện luôn đúng (tùy `nums[i] ≤ nums[i+1]` hay `>`) ⇒ chỉ số cuối **luôn** tới được, đáp án luôn hữu hạn, và `Σ costs[1..n−1]` là một cận trên.

## 4. Ý tưởng nền tảng / trực giác

**Điều kiện 1** (`nums[i] ≤ nums[j]`, mọi phần tử giữa `< nums[i]`): đi từ `i` sang phải, các phần tử giữa đều `< nums[i]`, và `j` là phần tử đầu tiên **không** `< nums[i]`, tức là phần tử **≥ `nums[i]` đầu tiên**. Nếu có `j' > j` cũng thỏa `nums[j'] ≥ nums[i]` thì `j` nằm giữa `i` và `j'` mà `nums[j] ≥ nums[i]` — vi phạm. ⇒ đúng **một** đích: `nge[i]`.

**Điều kiện 2** (`nums[i] > nums[j]`, mọi phần tử giữa `≥ nums[i]`): tương tự, `j` = phần tử **< `nums[i]` đầu tiên** ⇒ đúng một đích: `nlt[i]`.

⇒ Đồ thị có tối đa `2n` cạnh. Đáp án = đường đi ngắn nhất 0 → n−1, trọng số cạnh `(i, j)` là `costs[j]`.

Tính tay ví dụ 1: `nums = [3,2,4,4,1]`.

| i | nums | nge (≥ đầu tiên) | nlt (< đầu tiên) |
|---|---|---|---|
| 0 | 3 | 2 | 1 |
| 1 | 2 | 2 | 4 |
| 2 | 4 | 3 | 4 |
| 3 | 4 | − | 4 |
| 4 | 1 | − | − |

DP: `f[0] = 0`; từ 0: `f[2] = 6`, `f[1] = 7`; từ 1: `f[2] = min(6, 7 + 6)`, `f[4] = 7 + 2 = 9`; từ 2: `f[3] = 6 + 4 = 10`, `f[4] = min(9, 6 + 2) = 8`; từ 3: `f[4] = min(8, 10 + 2)` ⇒ **8** ✔.

In [4]:
a, c, _ = examples[0]
nge, nlt = next_ge(a), next_lt(a)
f = [inf] * len(a); f[0] = 0
for i in range(len(a)):
    for j in (nge[i], nlt[i]):
        if j != -1 and f[i] + c[j] < f[j]:
            print(f"  relax {i} -> {j}: f[{j}] = {f[i]} + {c[j]} = {f[i] + c[j]}"); f[j] = f[i] + c[j]
print("f =", f)

  relax 0 -> 2: f[2] = 0 + 6 = 6
  relax 0 -> 1: f[1] = 0 + 7 = 7
  relax 1 -> 4: f[4] = 7 + 2 = 9
  relax 2 -> 3: f[3] = 6 + 4 = 10
  relax 2 -> 4: f[4] = 6 + 2 = 8
f = [0, 7, 6, 10, 8]


## 5. Cách 1 — Brute Force / baseline

### 5.1 DP với kiểm tra điều kiện theo định nghĩa — O(n³)

In [5]:
def brute(nums, costs):
    n = len(nums); f = [inf] * n; f[0] = 0
    for j in range(1, n):
        for i in range(j):
            if can_jump(nums, i, j):
                f[j] = min(f[j], f[i] + costs[j])
    return f[n - 1]

for a, c, exp in examples:
    print(brute(a, c), "OK" if brute(a, c) == exp else "SAI")

8 OK
2 OK


### 5.2 Baseline 2 — Với mỗi `i`, quét sang phải tìm hai đích — O(n²) worst

Dùng nhận xét ở mục 4 nhưng tìm `nge`/`nlt` bằng vòng lặp. Worst case: mảng giảm dần chậm rồi tăng (đích ≥ ở rất xa).

In [6]:
def scan_targets(nums, costs):
    n = len(nums); f = [inf] * n; f[0] = 0
    for i in range(n):
        ge = lt = -1
        for j in range(i + 1, n):
            if ge == -1 and nums[j] >= nums[i]: ge = j
            if lt == -1 and nums[j] < nums[i]: lt = j
            if ge != -1 and lt != -1: break
        for j in (ge, lt):
            if j != -1: f[j] = min(f[j], f[i] + costs[j])
    return f[n - 1]

for a, c, exp in examples:
    print(scan_targets(a, c), "OK" if scan_targets(a, c) == exp else "SAI")

8 OK
2 OK


**Nút thắt:** tìm "phần tử kế tiếp thỏa điều kiện" lặp lại cho từng `i`. Bài toán "next greater / next smaller" cho mọi `i` là đất diễn của ngăn xếp đơn điệu — O(n) tổng cộng.

## 6. Ý tưởng thuật toán / bước đột phá

1. Tính `nge[i]` và `nlt[i]` bằng hai ngăn xếp đơn điệu (mỗi cái O(n)).
2. DP trên DAG: `f[0] = 0`; duyệt `i = 0 … n−1`, relax hai cạnh `i → nge[i]`, `i → nlt[i]` với trọng số `costs[đích]`.
3. Trả `f[n−1]`.

**Một lượt duy nhất (trái → phải):** khi đi tới `j`, những chỉ số `i < j` có `nge[i] = j` chính là những chỉ số bị **pop** khỏi ngăn xếp "chờ phần tử ≥" (ngăn xếp giảm ngặt) vì `nums[i] ≤ nums[j]`; tương tự những `i` có `nlt[i] = j` bị pop khỏi ngăn xếp "chờ phần tử <" (ngăn xếp không giảm) vì `nums[i] > nums[j]`. Mọi tiền nhiệm của `j` đều `< j` nên đã có `f` cuối cùng ⇒ tính được `f[j]` ngay lúc đó.

**Độ phức tạp:** O(n) thời gian, O(n) bộ nhớ.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hai ngăn xếp từ phải + DP (cách doocs)

In [7]:
def two_stacks_dp(nums, costs):
    n = len(nums); g = defaultdict(list)
    stk = []
    for i in range(n - 1, -1, -1):
        while stk and nums[stk[-1]] < nums[i]: stk.pop()
        if stk: g[i].append(stk[-1])
        stk.append(i)
    stk = []
    for i in range(n - 1, -1, -1):
        while stk and nums[stk[-1]] >= nums[i]: stk.pop()
        if stk: g[i].append(stk[-1])
        stk.append(i)
    f = [inf] * n; f[0] = 0
    for i in range(n):
        for j in g[i]:
            f[j] = min(f[j], f[i] + costs[j])
    return f[n - 1]

for a, c, exp in examples:
    print(two_stacks_dp(a, c), "OK" if two_stacks_dp(a, c) == exp else "SAI")

8 OK
2 OK


### 7.2 Cách 3 — Một lượt trái → phải, DP trực tuyến

In [8]:
def one_pass(nums, costs):
    n = len(nums); f = [inf] * n; f[0] = 0
    wait_ge = []                    # chỉ số chờ phần tử >= mình; nums giảm ngặt từ đáy lên
    wait_lt = []                    # chỉ số chờ phần tử <  mình; nums không giảm từ đáy lên
    for j in range(n):
        x = nums[j]; best = f[j]
        while wait_ge and nums[wait_ge[-1]] <= x:
            i = wait_ge.pop(); best = min(best, f[i] + costs[j])
        while wait_lt and nums[wait_lt[-1]] > x:
            i = wait_lt.pop(); best = min(best, f[i] + costs[j])
        f[j] = best
        wait_ge.append(j); wait_lt.append(j)
    return f[n - 1]

for a, c, exp in examples:
    print(one_pass(a, c), "OK" if one_pass(a, c) == exp else "SAI")

8 OK
2 OK


**Vì sao các chỉ số bị pop đúng là tiền nhiệm?** `wait_ge` chứa những `i` chưa gặp phần tử ≥ `nums[i]`. Chúng có `nums` **giảm ngặt** từ đáy lên (nếu `nums[i] ≤ nums[i']` với `i < i'` thì `i` đã bị `i'` pop). Phần tử mới `x` pop đúng những `i` có `nums[i] ≤ x` — những `i` lần đầu gặp phần tử ≥ mình — tức `nge[i] = j`. Lập luận tương tự cho `wait_lt`.

### 7.3 Cách 4 — Dijkstra trên đồ thị 2n cạnh (tổng quát hơn, không tận dụng DAG)

In [9]:
def dijkstra(nums, costs):
    n = len(nums); nge, nlt = next_ge(nums), next_lt(nums)
    dist = [inf] * n; dist[0] = 0; pq = [(0, 0)]
    while pq:
        d, i = heapq.heappop(pq)
        if d > dist[i]: continue
        if i == n - 1: return d
        for j in (nge[i], nlt[i]):
            if j != -1 and d + costs[j] < dist[j]:
                dist[j] = d + costs[j]; heapq.heappush(pq, (dist[j], j))
    return dist[n - 1]

for a, c, exp in examples:
    print(dijkstra(a, c), "OK" if dijkstra(a, c) == exp else "SAI")

8 OK
2 OK


Dijkstra đúng (trọng số không âm) nhưng O(n log n) và không cần thiết: đồ thị đã có thứ tự tô-pô tự nhiên. Nếu trọng số có thể **âm**, Dijkstra sai còn DP trên DAG vẫn đúng.

## 8. TRACE từng bước

### 8.1 Trace cách 3 (một lượt) trên ví dụ 1

In [10]:
def trace_one_pass(nums, costs):
    n = len(nums); f = [inf] * n; f[0] = 0; wait_ge, wait_lt = [], []
    for j in range(n):
        x = nums[j]; preds = []
        while wait_ge and nums[wait_ge[-1]] <= x:
            i = wait_ge.pop(); preds.append((i, "≥")); f[j] = min(f[j], f[i] + costs[j])
        while wait_lt and nums[wait_lt[-1]] > x:
            i = wait_lt.pop(); preds.append((i, "<")); f[j] = min(f[j], f[i] + costs[j])
        wait_ge.append(j); wait_lt.append(j)
        print(f"j={j} nums={x}: tiền nhiệm {preds!s:<22} f[{j}]={f[j]!s:<4} wait_ge={wait_ge} wait_lt={wait_lt}")
    return f[n - 1]

print("đáp án:", trace_one_pass(*examples[0][:2]))

j=0 nums=3: tiền nhiệm []                     f[0]=0    wait_ge=[0] wait_lt=[0]
j=1 nums=2: tiền nhiệm [(0, '<')]             f[1]=7    wait_ge=[0, 1] wait_lt=[1]
j=2 nums=4: tiền nhiệm [(1, '≥'), (0, '≥')]   f[2]=6    wait_ge=[2] wait_lt=[1, 2]
j=3 nums=4: tiền nhiệm [(2, '≥')]             f[3]=10   wait_ge=[3] wait_lt=[1, 2, 3]
j=4 nums=1: tiền nhiệm [(3, '<'), (2, '<'), (1, '<')] f[4]=8    wait_ge=[3, 4] wait_lt=[4]
đáp án: 8


Dòng `j = 4`: trước bước này `wait_lt = [1, 2, 3]` (giá trị `2, 4, 4`); phần tử `1` nhỏ hơn cả ba nên cả ba nhận chỉ số 4 làm "phần tử < đầu tiên" và đều là tiền nhiệm của 4. Chỉ số 0 đã rời `wait_lt` từ `j = 1` (vì `2 < 3`). Ở `j = 2`, giá trị 4 pop cả 1 và 0 khỏi `wait_ge` — nên 0 → 2 là một cạnh, đúng như bảng tính tay.

### 8.2 Trace ví dụ 2 — vì sao không nhảy thẳng 0 → 2

In [11]:
a, c, _ = examples[1]
print("nge:", next_ge(a), "| nlt:", next_lt(a))
print("can_jump(0, 2):", can_jump(a, 0, 2), "— nums[1] = 1 không < nums[0] = 0")
print("đáp án:", trace_one_pass(a, c))

nge: [1, 2, -1] | nlt: [-1, -1, -1]
can_jump(0, 2): False — nums[1] = 1 không < nums[0] = 0
j=0 nums=0: tiền nhiệm []                     f[0]=0    wait_ge=[0] wait_lt=[0]
j=1 nums=1: tiền nhiệm [(0, '≥')]             f[1]=1    wait_ge=[1] wait_lt=[0, 1]
j=2 nums=2: tiền nhiệm [(1, '≥')]             f[2]=2    wait_ge=[2] wait_lt=[0, 1, 2]
đáp án: 2


## 9. Code nộp LeetCode

In [12]:
from typing import List
from math import inf

class Solution:
    def minCost(self, nums: List[int], costs: List[int]) -> int:
        n = len(nums)
        f = [inf] * n
        f[0] = 0
        wait_ge = []        # chưa gặp phần tử >= mình (nums giảm ngặt từ đáy lên)
        wait_lt = []        # chưa gặp phần tử <  mình (nums không giảm từ đáy lên)
        for j, x in enumerate(nums):
            best = f[j]
            c = costs[j]
            while wait_ge and nums[wait_ge[-1]] <= x:     # j là phần tử >= đầu tiên của chúng
                i = wait_ge.pop()
                if f[i] + c < best:
                    best = f[i] + c
            while wait_lt and nums[wait_lt[-1]] > x:      # j là phần tử < đầu tiên của chúng
                i = wait_lt.pop()
                if f[i] + c < best:
                    best = f[i] + c
            f[j] = best
            wait_ge.append(j)
            wait_lt.append(j)
        return f[n - 1]

In [13]:
sol = Solution()
for a, c, exp in examples:
    got = sol.minCost(a, c)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

8 | kỳ vọng 8 | OK
2 | kỳ vọng 2 | OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (chỉ hai đích).** Từ `i` nhảy được tới `j` ⇔ `j = nge[i]` hoặc `j = nlt[i]`.

*(⇒)* Nếu nhảy theo điều kiện 1: `nums[j] ≥ nums[i]` và mọi `k ∈ (i, j)` có `nums[k] < nums[i]` ⇒ không có chỉ số nào trong `(i, j)` có `nums ≥ nums[i]` ⇒ `j` là chỉ số đầu tiên như vậy ⇒ `j = nge[i]`. Theo điều kiện 2: tương tự `j = nlt[i]`.

*(⇐)* `j = nge[i]`: mọi `k ∈ (i, j)` có `nums[k] < nums[i]` (vì `j` là chỉ số ≥ đầu tiên) và `nums[j] ≥ nums[i]` ⇒ điều kiện 1. `j = nlt[i]`: mọi `k ∈ (i, j)` có `nums[k] ≥ nums[i]` và `nums[j] < nums[i]` ⇒ điều kiện 2.

**Bổ đề 2 (DP trên DAG).** Cạnh chỉ đi từ chỉ số nhỏ sang lớn. Đặt `F(j)` = chi phí nhỏ nhất từ 0 tới `j`. `F(0) = 0`, `F(j) = min(F(i) + costs[j] : i → j)`. Quy nạp theo `j`: khi xử lý `j`, mọi tiền nhiệm `i < j` đã có `f[i] = F(i)` ⇒ `f[j] = F(j)`.

**Bổ đề 3 (ngăn xếp một lượt tìm đúng tiền nhiệm).** Bất biến của `wait_ge` trước khi xử lý `j`: chứa đúng các `i < j` mà **không** có `k ∈ (i, j)` với `nums[k] ≥ nums[i]`, theo thứ tự chỉ số tăng từ đáy lên, `nums` giảm ngặt. Khi thêm `j`: các `i` có `nums[i] ≤ nums[j]` có `nge[i] = j` — chúng nằm ở **đỉnh** (vì `nums` giảm ngặt từ đáy lên, các phần tử nhỏ nhất ở trên) nên vòng `while` pop đúng và đủ chúng; các `i` còn lại có `nums[i] > nums[j]` vẫn chờ, và `j` được đẩy lên — bất biến giữ (vì `nums[j] <` mọi phần tử còn lại). Tương tự cho `wait_lt` với `nums` không giảm từ đáy lên. ∎

**Hệ quả:** đáp án hữu hạn và `≤ Σ_{j≥1} costs[j]` (đường 0 → 1 → … → n−1).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. DP + kiểm tra định nghĩa | O(n³) | O(n) |
| 1b. Quét tìm hai đích | O(n²) | O(n) |
| 2. Hai ngăn xếp + DP (doocs) | O(n) | O(n) |
| 3. Một lượt | O(n) | O(n) |
| 4. Dijkstra | O(n log n) | O(n) |

In [14]:
rng = random.Random(2297)
N = 10**5
tests = {"ngẫu nhiên": ([rng.randint(0, 10**5) for _ in range(N)], [rng.randint(0, 10**5) for _ in range(N)]),
         "tăng dần": (list(range(N)), [rng.randint(0, 10**5) for _ in range(N)]),
         "giảm dần": (list(range(N, 0, -1)), [rng.randint(0, 10**5) for _ in range(N)]),
         "hằng": ([7] * N, [1] * N)}
for label, (a, c) in tests.items():
    res = []
    for name, f in [("two_stacks_dp", two_stacks_dp), ("one_pass", one_pass), ("dijkstra", dijkstra), ("Solution", Solution().minCost)]:
        t0 = time.perf_counter(); res.append(f(a, c)); dt = time.perf_counter() - t0
        print(f"{label:<10} {name:<13} {dt*1000:7.1f} ms -> {res[-1]}")
    assert len(set(res)) == 1

ngẫu nhiên two_stacks_dp    88.9 ms -> 2259734
ngẫu nhiên one_pass         28.6 ms -> 2259734
ngẫu nhiên dijkstra         89.9 ms -> 2259734
ngẫu nhiên Solution         27.8 ms -> 2259734


tăng dần   two_stacks_dp    89.4 ms -> 5017927454
tăng dần   one_pass         18.8 ms -> 5017927454
tăng dần   dijkstra         34.3 ms -> 5017927454
tăng dần   Solution         16.2 ms -> 5017927454
giảm dần   two_stacks_dp    39.9 ms -> 5005786687
giảm dần   one_pass         17.5 ms -> 5005786687
giảm dần   dijkstra         34.9 ms -> 5005786687
giảm dần   Solution         16.2 ms -> 5005786687


hằng       two_stacks_dp    39.1 ms -> 99999
hằng       one_pass         16.3 ms -> 99999
hằng       dijkstra         33.4 ms -> 99999
hằng       Solution         15.5 ms -> 99999


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng — đối chiếu lại đề gốc nếu có Premium)

In [15]:
ALL = {"brute": brute, "scan_targets": scan_targets, "two_stacks_dp": two_stacks_dp, "one_pass": one_pass,
       "dijkstra": dijkstra, "Solution": Solution().minCost}
edge = examples + [
    ([5], [9], 0),                                  # n = 1: đã ở đích, không tốn gì
    ([1, 2], [100, 3], 3),                          # costs[0] không bao giờ bị tính
    ([2, 2, 2, 2], [0, 5, 5, 5], 15),               # bằng nhau: chỉ nhảy sang kề bên (nge = i+1)
    ([3, 1, 2, 5], [0, 1, 100, 1], 1),              # nhảy thẳng 0 -> 3 (giữa là 1, 2 < 3)
    ([5, 1, 1, 1, 6], [0, 9, 9, 9, 1], 1),          # nhảy thẳng tới phần tử >= đầu tiên
    ([1, 5, 5, 5, 0], [0, 9, 9, 9, 1], 1),          # nhảy thẳng tới phần tử < đầu tiên
    ([0, 0, 0], [0, 0, 0], 0),
]
for a, c, exp in edge:
    ref = brute(a, c)
    for name, f in ALL.items():
        assert f(a, c) == ref, (name, a, c, f(a, c), ref)
    print(f"nums={a!s:<16} costs={c!s:<20} -> {ref} | ghi trong bảng: {exp} | khớp: {ref == exp}")

nums=[3, 2, 4, 4, 1]  costs=[3, 7, 6, 4, 2]      -> 8 | ghi trong bảng: 8 | khớp: True
nums=[0, 1, 2]        costs=[1, 1, 1]            -> 2 | ghi trong bảng: 2 | khớp: True
nums=[5]              costs=[9]                  -> 0 | ghi trong bảng: 0 | khớp: True
nums=[1, 2]           costs=[100, 3]             -> 3 | ghi trong bảng: 3 | khớp: True
nums=[2, 2, 2, 2]     costs=[0, 5, 5, 5]         -> 15 | ghi trong bảng: 15 | khớp: True
nums=[3, 1, 2, 5]     costs=[0, 1, 100, 1]       -> 1 | ghi trong bảng: 1 | khớp: True
nums=[5, 1, 1, 1, 6]  costs=[0, 9, 9, 9, 1]      -> 1 | ghi trong bảng: 1 | khớp: True
nums=[1, 5, 5, 5, 0]  costs=[0, 9, 9, 9, 1]      -> 1 | ghi trong bảng: 1 | khớp: True
nums=[0, 0, 0]        costs=[0, 0, 0]            -> 0 | ghi trong bảng: 0 | khớp: True


Cell trên luôn đối chiếu với vét cạn theo định nghĩa và in `khớp:` để phát hiện kỳ vọng ghi tay sai. Case `[3, 1, 2, 5]` đáng chú ý: từ 0 (giá trị 3) nhảy được tới 1 (phần tử `< 3` đầu tiên) **và** tới 3 (phần tử `≥ 3` đầu tiên, các phần tử giữa `1, 2` đều `< 3`). Nhảy thẳng 0 → 3 chỉ tốn 1, còn mọi đường qua chỉ số 1 tốn ít nhất `1 + 1 = 2`. Dễ nhầm là chỉ nhảy được tới phần tử kề bên — khi tính tay, lần đầu mình đã ghi kỳ vọng 2, và chính bước đối chiếu vét cạn đã bắt được lỗi đó.

### 12.2 Đối chiếu ngẫu nhiên với vét cạn (seed cố định)

In [16]:
rng = random.Random(42)
for _ in range(3000):
    n = rng.randint(1, 9)
    a = [rng.randint(0, rng.choice([2, 4, 10])) for _ in range(n)]; c = [rng.randint(0, 9) for _ in range(n)]
    ref = brute(a, c)
    for name, f in ALL.items():
        assert f(a, c) == ref, (name, a, c)
print("3000 bộ ngẫu nhiên OK")

3000 bộ ngẫu nhiên OK


### 12.3 Tính chất: đáp án ≤ Σ costs[1:], ≥ min cạnh vào đích; mỗi `i` có 1 hoặc 2 đích hợp lệ (trừ `n − 1`)

In [17]:
rng = random.Random(7)
for _ in range(2000):
    n = rng.randint(2, 12)
    a = [rng.randint(0, 5) for _ in range(n)]; c = [rng.randint(0, 9) for _ in range(n)]
    ans = Solution().minCost(a, c)
    assert c[n - 1] <= ans <= sum(c[1:])
    for i in range(n - 1):
        targets = [j for j in range(i + 1, n) if can_jump(a, i, j)]
        assert 1 <= len(targets) <= 2 and (i + 1) in targets
        assert set(targets) == {j for j in (next_ge(a)[i], next_lt(a)[i]) if j != -1}
print("tính chất OK")

tính chất OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Bỏ điều kiện "các phần tử ở giữa" (nhảy tới **mọi** `j` có `nums[j] ≥ nums[i]` hoặc `< nums[i]`)

In [18]:
def wrong_ignore_middle(nums, costs):
    n = len(nums); f = [inf] * n; f[0] = 0
    for j in range(1, n):
        f[j] = min(f[i] for i in range(j)) + costs[j]       # SAI: mọi cặp i < j đều "nhảy được"
    return f[n - 1]

for a, c, _ in examples:
    print("bản sai :", wrong_ignore_middle(a, c), "| bản đúng:", Solution().minCost(a, c))

bản sai : 2 | bản đúng: 8
bản sai : 1 | bản đúng: 2


**Vì sao sai:** hai điều kiện "≥" và "<" phủ mọi `j`, nên nếu bỏ phần "ở giữa" thì nhảy được tới bất kỳ đâu ⇒ đáp án luôn là `costs[n−1]`. Ví dụ 2 cho 1 thay vì 2.

### Lỗi 2 — Dùng phần tử **lớn hơn ngặt** đầu tiên cho điều kiện 1 (sai dấu khi bằng nhau)

In [19]:
def next_gt(nums):
    n = len(nums); res = [-1] * n; st = []
    for i in range(n - 1, -1, -1):
        while st and nums[st[-1]] <= nums[i]: st.pop()      # SAI: pop cả phần tử bằng
        res[i] = st[-1] if st else -1
        st.append(i)
    return res

def wrong_strict_greater(nums, costs):
    n = len(nums); g1, g2 = next_gt(nums), next_lt(nums); f = [inf] * n; f[0] = 0
    for i in range(n):
        for j in (g1[i], g2[i]):
            if j != -1: f[j] = min(f[j], f[i] + costs[j])
    return f[n - 1]

a, c = [2, 2, 3], [0, 1, 10]
print("bản sai :", wrong_strict_greater(a, c), "| bản đúng:", Solution().minCost(a, c))
a, c = [2, 2, 2], [0, 1, 1]
print("bản sai :", wrong_strict_greater(a, c), "| bản đúng:", Solution().minCost(a, c))

bản sai : 10 | bản đúng: 11
bản sai : inf | bản đúng: 2


**Vì sao sai:** `[2, 2, 3]`: từ 0 phải nhảy tới 1 (cũng là 2: `nums[0] ≤ nums[1]`, giữa rỗng); bản sai lại cho nhảy thẳng 0 → 2 (phần tử > 2 đầu tiên) dù ở giữa có `2` **không** `< 2` — nhỏ hơn chi phí thật. `[2, 2, 2]`: không có phần tử > 2 nên bản sai mất hết cạnh, không tới được đích (`inf`). **Sửa:** điều kiện 1 dùng `≥`.

### Lỗi 3 — Dùng phần tử **≤** đầu tiên cho điều kiện 2

In [20]:
def next_le(nums):
    n = len(nums); res = [-1] * n; st = []
    for i in range(n - 1, -1, -1):
        while st and nums[st[-1]] > nums[i]: st.pop()       # SAI: giữ cả phần tử bằng
        res[i] = st[-1] if st else -1
        st.append(i)
    return res

def wrong_le(nums, costs):
    n = len(nums); g1, g2 = next_ge(nums), next_le(nums); f = [inf] * n; f[0] = 0
    for i in range(n):
        for j in (g1[i], g2[i]):
            if j != -1: f[j] = min(f[j], f[i] + costs[j])
    return f[n - 1]

found = None; rng = random.Random(13)
for _ in range(20000):
    n = rng.randint(2, 6); a = [rng.randint(0, 3) for _ in range(n)]; c = [rng.randint(0, 9) for _ in range(n)]
    if wrong_le(a, c) != Solution().minCost(a, c): found = (a, c); break
a, c = found
print("phản ví dụ:", a, c, "| bản sai :", wrong_le(a, c), "| bản đúng:", Solution().minCost(a, c))
print("ví dụ đề:", [wrong_le(x, y) for x, y, _ in examples], "| đúng:", [e[2] for e in examples])

phản ví dụ: [1, 2, 0, 3, 1, 0] [4, 2, 1, 4, 7, 6] | bản sai : 7 | bản đúng: 18
ví dụ đề: [9, 2] | đúng: [8, 2]


**Vì sao sai:** phần tử **bằng** `nums[i]` thuộc điều kiện 1 (đã được `nge` xử lý), không thuộc điều kiện 2. Khi `nge[i]` và "phần tử ≤ đầu tiên" trùng nhau (cùng là phần tử bằng), bản sai **mất** cạnh tới phần tử `<` đầu tiên thật sự. Ngược lại, khi giữa `i` và phần tử bằng có một phần tử **lớn hơn**, bản sai còn **thêm** một cạnh không hợp lệ (tới phần tử bằng, nhảy qua phần tử lớn hơn) — vì vậy bản sai có thể cho kết quả lớn hơn **hoặc nhỏ hơn** đáp án thật (phản ví dụ bên dưới cho 7 < 18). Ví dụ 1 có giá trị lặp (`4, 4`) nên đã lộ lỗi: từ chỉ số 2, "phần tử ≤ 4 đầu tiên" là chỉ số 3 — trùng với `nge[2]` — nên mất cạnh 2 → 4, đáp án thành 9. Ví dụ 2 không có giá trị lặp nên **trùng hợp** đúng. Phản ví dụ thứ hai được tìm bằng vét cạn ngẫu nhiên.

### Lỗi 4 — Cộng cả `costs[0]` (chi phí của điểm xuất phát)

In [21]:
def wrong_start_cost(nums, costs):
    return Solution().minCost(nums, costs) + costs[0]          # SAI: không "nhảy tới" chỉ số 0

for a, c, _ in examples:
    print("bản sai :", wrong_start_cost(a, c), "| bản đúng:", Solution().minCost(a, c))

bản sai : 11 | bản đúng: 8
bản sai : 3 | bản đúng: 2


### Lỗi 5 — Tham lam: luôn nhảy tới đích **rẻ hơn** trong hai đích

In [22]:
def wrong_greedy(nums, costs):
    n = len(nums); nge, nlt = next_ge(nums), next_lt(nums); i = 0; total = 0
    while i != n - 1:
        cands = [j for j in (nge[i], nlt[i]) if j != -1]
        j = min(cands, key=lambda j: costs[j])                  # SAI: rẻ trước mắt chưa chắc rẻ về sau
        total += costs[j]; i = j
    return total

for a, c, _ in examples:
    print("bản sai :", wrong_greedy(a, c), "| bản đúng:", Solution().minCost(a, c))

bản sai : 8 | bản đúng: 8
bản sai : 2 | bản đúng: 2


**Vì sao sai:** cả hai ví dụ của đề đều **trùng hợp** đúng: ở ví dụ 1, từ 0 đích rẻ hơn là 2 (`6 < 7`), rồi từ 2 đích rẻ hơn là 4 (`2 < 4`) — đúng là đường tối ưu. Ở ví dụ 2 mỗi chỉ số chỉ có một đích. Phản ví dụ dưới được tìm bằng vét cạn.

In [23]:
found = None; rng = random.Random(17)
for _ in range(20000):
    n = rng.randint(2, 7); a = [rng.randint(0, 4) for _ in range(n)]; c = [rng.randint(0, 9) for _ in range(n)]
    if wrong_greedy(a, c) != Solution().minCost(a, c): found = (a, c); break
a, c = found
print("phản ví dụ:", a, c, "| bản sai :", wrong_greedy(a, c), "| bản đúng:", Solution().minCost(a, c))

phản ví dụ: [3, 2, 2, 2, 1, 4] [4, 1, 0, 3, 6, 6] | bản sai : 10 | bản đúng: 6


**Sửa:** DP trên toàn bộ đồ thị — một bước rẻ có thể dẫn vào đoạn đường đắt.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Điều kiện "mọi phần tử ở giữa < / ≥ …" ⇒ thường chỉ ra **phần tử kế tiếp lớn hơn / nhỏ hơn** ⇒ ngăn xếp đơn điệu, số cạnh O(n).
- Chỉ nhảy sang phải ⇒ **DAG theo chỉ số** ⇒ DP xuôi, không cần Dijkstra.
- Phân biệt kỹ `≥` / `>` và `<` / `≤` — viết vét cạn theo định nghĩa và tìm phản ví dụ ngẫu nhiên với giá trị nhỏ (nhiều phần tử bằng nhau).
- Có thể gộp "tìm next greater" và DP thành một lượt trái → phải: phần tử bị pop biết ngay "người kế tiếp" của nó.
- Bài cùng họ: 1340 Jump Game V, 975 Odd Even Jump, 1696 Jump Game VI, 2282 Number of People That Can Be Seen in a Grid, 496/503 Next Greater Element, 1944.

## 15. Biến thể của bài

### 15.1 Biến thể A — Trả về **đường đi** chi phí nhỏ nhất (lưu cha)

In [24]:
def min_cost_path(nums, costs):
    n = len(nums); f = [inf] * n; f[0] = 0; par = [-1] * n
    nge, nlt = next_ge(nums), next_lt(nums)
    for i in range(n):
        for j in (nge[i], nlt[i]):
            if j != -1 and f[i] + costs[j] < f[j]:
                f[j] = f[i] + costs[j]; par[j] = i
    path = [n - 1]
    while path[-1] != 0: path.append(par[path[-1]])
    return f[n - 1], path[::-1]

for a, c, _ in examples:
    v, p = min_cost_path(a, c)
    assert all(can_jump(a, x, y) for x, y in zip(p, p[1:])) and sum(c[j] for j in p[1:]) == v
    print(v, p)

8 [0, 2, 4]
2 [0, 1, 2]


### 15.2 Biến thể B — Đếm số đường đi chi phí nhỏ nhất (mod 10⁹+7)

In [25]:
def count_min_paths(nums, costs, MOD=10**9 + 7):
    n = len(nums); f = [inf] * n; f[0] = 0; cnt = [0] * n; cnt[0] = 1
    nge, nlt = next_ge(nums), next_lt(nums)
    for i in range(n):
        for j in {nge[i], nlt[i]} - {-1}:
            v = f[i] + costs[j]
            if v < f[j]: f[j], cnt[j] = v, cnt[i]
            elif v == f[j]: cnt[j] = (cnt[j] + cnt[i]) % MOD
    return cnt[n - 1]

def count_min_paths_brute(nums, costs):
    n = len(nums); best = [inf]; ways = [0]
    def go(i, cost):
        if i == n - 1:
            if cost < best[0]: best[0], ways[0] = cost, 1
            elif cost == best[0]: ways[0] += 1
            return
        for j in range(i + 1, n):
            if can_jump(nums, i, j): go(j, cost + costs[j])
    go(0, 0); return ways[0]

rng = random.Random(152)
for _ in range(1000):
    n = rng.randint(1, 8); a = [rng.randint(0, 3) for _ in range(n)]; c = [rng.randint(0, 2) for _ in range(n)]
    assert count_min_paths(a, c) == count_min_paths_brute(a, c)
print(count_min_paths([0, 0, 0, 0], [0, 0, 0, 0]), count_min_paths(*examples[0][:2]))

1 1


### 15.3 Biến thể C — Chi phí có thể **âm**: DP trên DAG vẫn đúng, Dijkstra thì không

In [26]:
def dag_dp(nums, costs):
    n = len(nums); nge, nlt = next_ge(nums), next_lt(nums); f = [inf] * n; f[0] = 0
    for i in range(n):
        for j in (nge[i], nlt[i]):
            if j != -1: f[j] = min(f[j], f[i] + costs[j])
    return f[n - 1]

bad = None; rng = random.Random(153)
for _ in range(20000):
    n = rng.randint(2, 7); a = [rng.randint(0, 4) for _ in range(n)]; c = [rng.randint(-9, 9) for _ in range(n)]
    assert dag_dp(a, c) == brute(a, c)
    if bad is None and dijkstra(a, c) != brute(a, c): bad = (a, c)
print("DP trên DAG luôn khớp vét cạn; phản ví dụ cho Dijkstra:", bad, "->", dijkstra(*bad), "vs", brute(*bad))

DP trên DAG luôn khớp vét cạn; phản ví dụ cho Dijkstra: ([3, 1, 1, 1, 3, 0], [-4, 6, -9, 5, 5, -4]) -> 1 vs -7


### 15.4 Biến thể D — Số bước nhảy **ít nhất** (mọi chi phí bằng 1)

Đặt `costs = [1] * n`; hoặc BFS trên đồ thị 2n cạnh.

In [27]:
def min_jumps(nums):
    return Solution().minCost(nums, [1] * len(nums))
print(min_jumps([3, 2, 4, 4, 1]), min_jumps([0, 1, 2]), min_jumps([5, 1, 1, 1, 6]))

2 2 1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DP + định nghĩa | kiểm tra mọi cặp | O(n³) | O(n) | kiểm chứng |
| 1b. Quét tìm 2 đích | dùng bổ đề 1 | O(n²) | O(n) | trung gian |
| 2. Hai ngăn xếp + DP | doocs | O(n) | O(n) | rõ ràng, tách bước |
| 3. Một lượt | pop = tìm thấy đích | O(n) | O(n) | **mặc định**, gọn |
| 4. Dijkstra | đồ thị tổng quát | O(n log n) | O(n) | khi cạnh không theo thứ tự |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `minCost([1, 1, 1], [5, 5, 5])` và `minCost([4, 3, 2, 1], [0, 9, 9, 1])`?

**Bài 2 (tính tay):** Tính `nge`, `nlt` và `f` cho `nums = [2, 5, 1, 5, 3]`, `costs = [0, 4, 2, 1, 3]`.

**Bài 3 (điền TODO):** Viết `bt3_todo(nums)` trả về danh sách cạnh `(i, j)` của đồ thị nhảy (đã sắp xếp), O(n).

**Bài 4 (tự viết lại):** Viết `bt4_todo(nums, costs)` cài cách **hai ngăn xếp + DP** không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(nums, costs)` trả về chi phí nhỏ nhất **từ mọi chỉ số `s`** tới `n − 1` (danh sách độ dài `n`), O(n).

In [28]:
def bt3_todo(nums):
    # TODO: danh sách cạnh
    pass

def bt4_todo(nums, costs):
    # TODO: hai ngăn xếp + DP
    pass

def bt5_todo(nums, costs):
    # TODO: chi phí từ mọi s tới n-1
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: toàn bằng nhau ⇒ `nge[i] = i + 1`, `nlt` không có. Mức 2: phải đi từng bước ⇒ `5 + 5 = 10`. Mức 3: giảm dần ⇒ `nlt[0] = 1`, không có `nge` ⇒ đi từng bước `9 + 9 + 1 = 19`.

**Bài 2.** Mức 1: `nge = [1, 3, 3, −, −]`, `nlt = [2, 2, −, 4, −]`. Mức 2: `f[1] = 4`, `f[2] = 2` (từ 0), `f[3] = min(4 + 1, 2 + 1) = 3`. Mức 3: `f[4] = f[3] + 3 = 6`.

**Bài 3.** Mức 1: tính `nge`, `nlt`. Mức 2: thêm `(i, nge[i])`, `(i, nlt[i])` nếu khác −1. Mức 3: sắp xếp.

**Bài 4.** Mức 1: ngăn xếp từ phải, pop `<` ⇒ `nge`. Mức 2: pop `≥` ⇒ `nlt`. Mức 3: DP xuôi.

**Bài 5.** Mức 1: đảo chiều DP: `g[i]` = chi phí nhỏ nhất từ `i` tới `n − 1`. Mức 2: `g[n−1] = 0`, `g[i] = min(costs[j] + g[j])` trên hai đích `j`. Mức 3: duyệt `i` giảm dần.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `10`, `19`. **Bài 2:** `f = [0, 4, 2, 3, 6]`, đáp án `6`.

In [29]:
assert Solution().minCost([1, 1, 1], [5, 5, 5]) == 10
assert Solution().minCost([4, 3, 2, 1], [0, 9, 9, 1]) == 19
assert next_ge([2, 5, 1, 5, 3]) == [1, 3, 3, -1, -1] and next_lt([2, 5, 1, 5, 3]) == [2, 2, -1, 4, -1]
assert Solution().minCost([2, 5, 1, 5, 3], [0, 4, 2, 1, 3]) == 6

def bt3(nums):
    nge, nlt = next_ge(nums), next_lt(nums)
    return sorted((i, j) for i in range(len(nums)) for j in (nge[i], nlt[i]) if j != -1)

def bt4(nums, costs):
    return two_stacks_dp(nums, costs)

def bt5(nums, costs):
    n = len(nums); nge, nlt = next_ge(nums), next_lt(nums); g = [inf] * n; g[n - 1] = 0
    for i in range(n - 2, -1, -1):
        g[i] = min(costs[j] + g[j] for j in (nge[i], nlt[i]) if j != -1)
    return g

rng = random.Random(1991)
for _ in range(1500):
    n = rng.randint(1, 8); a = [rng.randint(0, 4) for _ in range(n)]; c = [rng.randint(0, 9) for _ in range(n)]
    assert bt3(a) == [(i, j) for i in range(n) for j in range(i + 1, n) if can_jump(a, i, j)]
    assert bt4(a, c) == brute(a, c)
    assert bt5(a, c) == [brute(a[s:], [0] + c[s + 1:]) for s in range(n)]
print(bt3([3, 2, 4, 4, 1]), bt5(*examples[0][:2]))
print("Đáp án bài tập: tất cả assert OK")

[(0, 1), (0, 2), (1, 2), (1, 4), (2, 3), (2, 4), (3, 4)] [8, 2, 2, 2, 0]
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
f = [inf] * n; f[0] = 0
wait_ge, wait_lt = [], []                  # chờ phần tử >= mình / < mình
for j, x in enumerate(nums):
    while wait_ge and nums[wait_ge[-1]] <= x: f[j] = min(f[j], f[wait_ge.pop()] + costs[j])
    while wait_lt and nums[wait_lt[-1]] >  x: f[j] = min(f[j], f[wait_lt.pop()] + costs[j])
    wait_ge.append(j); wait_lt.append(j)
return f[n - 1]
```

**Checklist:**
- [ ] Từ mỗi `i` chỉ có 2 đích: `≥` đầu tiên và `<` đầu tiên.
- [ ] Dấu: điều kiện 1 là `≥` (bằng nhau tính vào đây), điều kiện 2 là `<` ngặt.
- [ ] Không cộng `costs[0]`; `n = 1` ⇒ 0.
- [ ] DP theo chỉ số tăng (DAG), không tham lam.
- [ ] (Premium) Đối chiếu lại đề gốc.